In [1]:
# using pretrained VIT 
import sys
# !{sys.executable} -m pip install Transformers

In [2]:
import torch
import torch.nn as nn
from transformers import ViTModel, ViTConfig


In [3]:
# Config
NUM_CLASSES = 20
NUM_FRAMES = 16
IMG_SIZE = 224

In [4]:
class ASLViT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(ASLViT, self).__init__()
        
        # pretrained ViT backbone
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        
        # temporal pooling -- averages across all 16 frames
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        
        # classification head
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        # x shape: (batch, num_frames, C, H, W)
        batch_size, num_frames, C, H, W = x.shape
        
        # reshape to process all frames through ViT independently
        x = x.view(batch_size * num_frames, C, H, W)# (2, 16, 3, 224, 224) → (32, 3, 224, 224)
        
        # extract features from ViT for each frame
        outputs = self.vit(pixel_values=x)
        features = outputs.last_hidden_state[:, 0, :] # CLS token (batch*frames, 768)
        
        # reshape back and pool across frames
        features = features.view(batch_size, num_frames, -1)  # (32, 768) → (2, 16, 768)
        features = features.permute(0, 2, 1)                  # (2, 16, 768) → (2, 768, 16)
        features = self.temporal_pool(features).squeeze(-1)   # (2, 768, 16) → (2, 768)
        return self.classifier(features)

In [5]:
model = ASLViT(num_classes=NUM_CLASSES)
dummy_input = torch.randn(2, NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE)
output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Expected output: (2, {NUM_CLASSES})")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Input shape: torch.Size([2, 16, 3, 224, 224])
Output shape: torch.Size([2, 20])
Expected output: (2, 20)
